# LangGraph Agentic RAG — SEC EDGAR v2

**Changes from v1:**
- **Dataset**: Uses local `sec_chunks.jsonl` (12 companies, 155 filings, 2023–2026) instead of FinanceBench
- **Proposal 1 — Query Decomposition**: A `Planner` node decomposes multi-period / cross-company questions into 2–3 sub-queries, each with optional `ticker`/`filing_year` metadata filters for targeted retrieval
- **Proposal 2 — Refined Critic**: `CriticOutput.decision` now has a third state `insufficient_data` — the agent safely exits when required data is absent from the filing rather than looping

In [1]:
import os
import re
import time
import json
import random
import warnings
import threading
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple
from collections import deque, Counter

import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

from pydantic import BaseModel, Field, field_validator, model_validator

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings("ignore")

c:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = input("Enter your GROQ_API_KEY (leave blank if using Gemini): ").strip()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = input("Enter your GOOGLE_API_KEY (leave blank if using Groq): ").strip()

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

In [3]:
CONFIG: Dict[str, Any] = {
    "random_seed": 42,

    # ── Pilot vs full run ────────────────────────────────────────────────────
    "use_pilot":         False,   # True = pilot (5 questions); False = full (15 questions)
    "pilot_n_questions": 5,
    "full_n_questions":  15,
    "pilot_judge_sample_n": 1,
    "full_judge_sample_n":  2,

    # 'dev' (cheaper/faster) or 'final' (best quality)
    "execution_profile": "final",

    # SEC dataset paths
    "sec_chunks_path": r"C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl",
    "chroma_db_path":  r"C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\chroma_db",

    # retrieval / agent loop controls
    "max_context_retries": 1,
    "max_repair_retrievals": 1,
    "bm25_top_k": 8,
    "dense_top_k": 8,
    "rerank_top_k": 5,

    # decomposition: chunks retrieved per sub-query before merging
    "decomposition_top_k_per_subquery": 4,

    # Embedding model MUST match what was used to build the ChromaDB collection (768-dim)
    "dense_model_name":    "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",

    # Provider switch: 'groq' or 'gemini'
    "provider": "groq",

    # Groq models
    "groq_dev_generator_model":   "llama-3.1-8b-instant",
    "groq_dev_agent_model":       "llama-3.1-8b-instant",
    "groq_dev_judge_model":       "llama-3.1-8b-instant",
    "groq_final_generator_model": "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_final_agent_model":     "llama-3.1-8b-instant",
    "groq_final_judge_model":     "llama-3.1-8b-instant",
    "groq_fallback_agent_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],
    "groq_fallback_generator_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],
    "groq_fallback_judge_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],

    # Gemini models
    "gemini_dev_generator_model":   "gemini-2.0-flash-lite",
    "gemini_dev_agent_model":       "gemini-2.0-flash-lite",
    "gemini_dev_judge_model":       "gemini-2.0-flash-lite",
    "gemini_final_generator_model": "gemini-2.0-flash",
    "gemini_final_agent_model":     "gemini-2.0-flash",
    "gemini_final_judge_model":     "gemini-2.0-flash",

    # temperatures
    "temperature_planner":      0.0,
    "temperature_rewriter":     0.2,
    "temperature_context_eval": 0.0,
    "temperature_generator":    0.2,
    "temperature_critic":       0.0,
    "temperature_repair":       0.1,
    "temperature_judge":        0.0,
    # context formatting caps to reduce token usage in control nodes
    "generator_max_context_chunks": 8,
    "generator_max_context_chars": 12000,
    "control_max_context_chunks": 4,
    "control_max_context_chars": 6000,
    "judge_max_context_chunks": 3,
    "judge_max_context_chars": 4000,
    "enable_retrieval_sanity_check": True,

    # rate limiting — stay slightly under Groq's hard 30 RPM cap
    "max_rpm": 28,

    # inter-question sleep
    "inter_question_sleep_sec": 0.5,

    # retry settings
    "llm_max_retries": 3,
    "llm_retry_base_delay_sec": 5,

    # LLM judge sample size (per pipeline) — smaller for pilot
}

CONFIG["max_rpm"] = 28 if CONFIG["provider"] == "groq" else 10
CONFIG["judge_sample_n"] = CONFIG["pilot_judge_sample_n"] if CONFIG["use_pilot"] else CONFIG["full_judge_sample_n"]


def resolve_model_name(role: str) -> str:
    provider = CONFIG["provider"]
    profile  = CONFIG["execution_profile"]
    return CONFIG[f"{provider}_{profile}_{role}_model"]


def resolve_fallback_model_names(role: str) -> List[str]:
    provider = CONFIG["provider"]
    primary = resolve_model_name(role)
    fallbacks = CONFIG.get(f"{provider}_fallback_{role}_models", [])
    ordered = [primary] + list(fallbacks)
    return list(dict.fromkeys([m for m in ordered if m]))

In [4]:
# ---------------------------------------------------------------------------
# Evaluation question set — 15 questions across 12 companies
# Mix of single-period, multi-period, and cross-company questions.
# Multi-period and cross-company questions are expected to trigger decomposition.
# reference_answer is approximate — LLM judge is the primary quality metric.
# ---------------------------------------------------------------------------

SEC_EVAL_QUESTIONS = [
    # ── Single-period, single-company ────────────────────────────────────────
    {
        "id": "SEC_001",
        "question": "What was Apple's total net sales for fiscal year 2024?",
        "company": "Apple Inc.",
        "question_type": "single-period",
        "reference_answer": "$391.0 billion",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_002",
        "question": "What was Microsoft's total revenue for fiscal year 2024?",
        "company": "Microsoft Corporation",
        "question_type": "single-period",
        "reference_answer": "$245.1 billion",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_003",
        "question": "What was NVIDIA's total revenue for fiscal year 2025?",
        "company": "NVIDIA Corporation",
        "question_type": "single-period",
        "reference_answer": "$130.5 billion",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_004",
        "question": "What was JPMorgan Chase's total net revenue for fiscal year 2024?",
        "company": "JPMorgan Chase",
        "question_type": "single-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_005",
        "question": "What is UnitedHealth Group's medical care ratio for fiscal year 2024?",
        "company": "UnitedHealth Group",
        "question_type": "single-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_006",
        "question": "What was Costco's total membership fee revenue for fiscal year 2024?",
        "company": "Costco Wholesale",
        "question_type": "single-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    # ── Multi-period comparisons (expect decomposition) ───────────────────────
    {
        "id": "SEC_007",
        "question": "How did Apple's gross margin percentage change between fiscal year 2023 and fiscal year 2024?",
        "company": "Apple Inc.",
        "question_type": "multi-period",
        "reference_answer": "Gross margin was approximately 44.1% in FY2023 and 46.2% in FY2024",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_008",
        "question": "How did NVIDIA's data center revenue compare between fiscal year 2024 and fiscal year 2025?",
        "company": "NVIDIA Corporation",
        "question_type": "multi-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_009",
        "question": "Compare Microsoft's operating income between fiscal year 2023 and fiscal year 2024.",
        "company": "Microsoft Corporation",
        "question_type": "multi-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_010",
        "question": "How did Bank of America's net income compare between 2023 and 2024?",
        "company": "Bank of America",
        "question_type": "multi-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_011",
        "question": "Did ExxonMobil's capital expenditure increase or decrease from 2023 to 2024?",
        "company": "ExxonMobil",
        "question_type": "multi-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_012",
        "question": "How did Walmart's net sales change from fiscal year 2024 to fiscal year 2025?",
        "company": "Walmart",
        "question_type": "multi-period",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    # ── Cross-company comparisons (expect decomposition) ─────────────────────
    {
        "id": "SEC_013",
        "question": "Which company had higher total revenue in fiscal year 2024: Apple or Microsoft?",
        "company": "Apple Inc. / Microsoft",
        "question_type": "cross-company",
        "reference_answer": "Apple had higher revenue at approximately $391 billion versus Microsoft at $245 billion",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_014",
        "question": "Between JPMorgan Chase and Bank of America, which had the higher net income in 2024?",
        "company": "JPMorgan Chase / Bank of America",
        "question_type": "cross-company",
        "reference_answer": "",
        "expected_decision": "answer",
    },
    {
        "id": "SEC_015",
        "question": "Compare Chevron's and ExxonMobil's operating cash flow for fiscal year 2024.",
        "company": "Chevron / ExxonMobil",
        "question_type": "cross-company",
        "reference_answer": "",
        "expected_decision": "answer",
    },
]

# Pilot question IDs — hand-picked for coverage: 2 single-period + 2 multi-period + 1 cross-company
PILOT_IDS = ["SEC_001", "SEC_003", "SEC_007", "SEC_008", "SEC_013"]

full_eval_df = pd.DataFrame(SEC_EVAL_QUESTIONS)

if CONFIG["use_pilot"]:
    eval_df = full_eval_df[full_eval_df["id"].isin(PILOT_IDS)].reset_index(drop=True)
    print(f"PILOT RUN: {len(eval_df)} questions")
else:
    eval_df = full_eval_df.copy()
    print(f"FULL RUN: {len(eval_df)} questions")

print(eval_df[["id", "question_type", "company"]].to_string(index=False))

FULL RUN: 15 questions
     id question_type                          company
SEC_001 single-period                       Apple Inc.
SEC_002 single-period            Microsoft Corporation
SEC_003 single-period               NVIDIA Corporation
SEC_004 single-period                   JPMorgan Chase
SEC_005 single-period               UnitedHealth Group
SEC_006 single-period                 Costco Wholesale
SEC_007  multi-period                       Apple Inc.
SEC_008  multi-period               NVIDIA Corporation
SEC_009  multi-period            Microsoft Corporation
SEC_010  multi-period                  Bank of America
SEC_011  multi-period                       ExxonMobil
SEC_012  multi-period                          Walmart
SEC_013 cross-company           Apple Inc. / Microsoft
SEC_014 cross-company JPMorgan Chase / Bank of America
SEC_015 cross-company             Chevron / ExxonMobil


In [5]:
# ---------------------------------------------------------------------------
# SEC corpus loading
# ---------------------------------------------------------------------------

def load_sec_chunks(path: str) -> pd.DataFrame:
    """Load sec_chunks.jsonl and filter out low-value chunks."""
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    df = pd.DataFrame(chunks)
    before = len(df)
    df = df[~df["flag_low_value_combined"].fillna(False)].reset_index(drop=True)
    print(
        f"Loaded {before} total chunks → {len(df)} usable after filtering low-value "
        f"({before - len(df)} removed)"
    )
    print(
        f"Companies: {sorted(df['ticker'].unique())}  |  "
        f"Filing years: {sorted(df['filing_year'].unique())}"
    )
    return df


def sec_df_to_chunk_dicts(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Convert SEC DataFrame to the chunk dict format expected by CorpusIndex."""
    chunks = []
    for i, row in df.iterrows():
        doc_name = f"{row['ticker']}_{row['form_type']}_{str(row['filing_date'])[:10]}"
        contextual_text = (
            f"Company: {row['company_name']} ({row['ticker']})\n"
            f"Filing: {row['form_type']} | Date: {str(row['filing_date'])[:10]} | Year: {row['filing_year']}\n"
            f"Section: {row['section_title']}\n"
            f"Content: {row['text']}"
        )
        chunks.append({
            "chunk_id":     i,                  # integer row index (BM25 lookup)
            "chunk_id_str": str(row["chunk_id"]), # original string ID (ChromaDB cross-ref)
            "ticker":       row["ticker"],
            "company":      row["company_name"],
            "doc_name":     doc_name,
            "filing_year":  int(row["filing_year"]),
            "form_type":    row["form_type"],
            "page_num":     int(row.get("chunk_index", 0)),
            "raw_chunk":    row["text"],
            "contextual_chunk": contextual_text,
        })
    return chunks


print("Loading SEC chunks…")
sec_df = load_sec_chunks(CONFIG["sec_chunks_path"])

Loading SEC chunks…
Loaded 13152 total chunks → 12606 usable after filtering low-value (546 removed)
Companies: ['AAPL', 'BAC', 'BRK-B', 'COST', 'CVX', 'JNJ', 'JPM', 'MSFT', 'NVDA', 'UNH', 'WMT', 'XOM']  |  Filing years: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [6]:
# dense_model: encodes queries at search time (single vector — fast)
# ChromaDB holds the pre-built document embeddings so no all-chunk encoding needed
print("Loading dense embedder (query-time use only)…")
dense_model = SentenceTransformer(CONFIG["dense_model_name"])
print("Loading reranker…")
reranker = CrossEncoder(CONFIG["reranker_model_name"])
print("Models ready.")

Loading dense embedder (query-time use only)…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker…


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready.


In [7]:
# ---------------------------------------------------------------------------
# RetrievedChunk + CorpusIndex
#
# Dense search: ChromaDB (pre-built, loaded in ~1s — no re-encoding at startup)
# Keyword search: BM25Okapi (built from text in-memory, fast)
# Reranking: CrossEncoder (query-time, on merged candidates)
#
# Metadata filtering (ticker, filing_year, form_type) applies to BOTH:
#   BM25  → numpy score mask on self.df
#   Dense → ChromaDB `where` clause pushed to the vector store
# ---------------------------------------------------------------------------

@dataclass
class RetrievedChunk:
    doc_name: str
    company: str
    page_num: int
    chunk_id: str       # string chunk_id — consistent key for BM25 + ChromaDB dedup
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank + 1)


class CorpusIndex:
    def __init__(self, chunks: List[Dict[str, Any]], chroma_path: str):
        self.df = pd.DataFrame(chunks).copy()

        # BM25 — built from contextual text, fast (no ML)
        print("  Building BM25 index…")
        self.df["bm25_tokens"] = self.df["contextual_chunk"].str.lower().str.split()
        self.bm25 = BM25Okapi(self.df["bm25_tokens"].tolist())

        # String chunk_id → df integer row index (for BM25 chunk materialisation)
        self._str_to_row: Dict[str, int] = {
            row["chunk_id_str"]: idx for idx, row in self.df.iterrows()
        }

        # ChromaDB — load pre-built collection (embeddings already stored)
        print("  Loading ChromaDB collection…")
        client = chromadb.PersistentClient(path=chroma_path)
        collections = client.list_collections()
        if not collections:
            raise RuntimeError(f"No ChromaDB collections found at {chroma_path!r}")
        self.chroma_col = client.get_collection(collections[0].name)
        print(
            f"  ChromaDB ready: '{collections[0].name}'  "
            f"({self.chroma_col.count():,} docs)"
        )
        print(f"  CorpusIndex ready: {len(self.df):,} BM25 chunks + ChromaDB dense index")

    # ── Chunk materialisation helpers ────────────────────────────────────────

    def _chunk_from_row(self, row_idx: int, score: float, source: str) -> RetrievedChunk:
        row = self.df.iloc[row_idx]
        return RetrievedChunk(
            doc_name=row["doc_name"],
            company=row["company"],
            page_num=int(row.get("page_num", 0)),
            chunk_id=row["chunk_id_str"],
            raw_chunk=row["raw_chunk"],
            contextual_chunk=row["contextual_chunk"],
            score=float(score),
            source=source,
        )

    @staticmethod
    def _contextual_from_meta(text: str, meta: dict) -> str:
        return (
            f"Company: {meta['company_name']} ({meta['ticker']})\n"
            f"Filing: {meta['form_type']} | Date: {str(meta['filing_date'])[:10]} "
            f"| Year: {meta['filing_year']}\n"
            f"Section: {meta['section_title']}\n"
            f"Content: {text}"
        )

    def _chunk_from_chroma(self, doc_id: str, text: str, meta: dict, dist: float) -> RetrievedChunk:
        doc_name = f"{meta['ticker']}_{meta['form_type']}_{str(meta['filing_date'])[:10]}"
        return RetrievedChunk(
            doc_name=doc_name,
            company=meta["company_name"],
            page_num=int(meta.get("chunk_index", 0)),
            chunk_id=doc_id,                              # same string as BM25 chunk_id_str
            raw_chunk=text,
            contextual_chunk=self._contextual_from_meta(text, meta),
            score=1.0 - float(dist),                      # cosine distance → similarity
            source="chroma_dense",
        )

    # ── Metadata filter builders ─────────────────────────────────────────────

    def _bm25_mask(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[np.ndarray]:
        """Float mask (1.0/0.0) for BM25 score array. None = no filter."""
        if not ticker and not filing_year and not form_type:
            return None
        mask = np.ones(len(self.df), dtype=float)
        if ticker:
            mask *= (self.df["ticker"].str.upper() == ticker.upper()).astype(float).values
        if filing_year:
            mask *= (self.df["filing_year"] == int(filing_year)).astype(float).values
        if form_type:
            mask *= (self.df["form_type"].str.upper() == form_type.upper()).astype(float).values
        if mask.sum() < 5:
            print(f"  [Warn] BM25 filter matched < 5 chunks — using full corpus")
            return None
        return mask

    def _chroma_where(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[dict]:
        """ChromaDB `where` clause for metadata filtering. None = no filter."""
        conditions = []
        if ticker:
            conditions.append({"ticker": {"$eq": ticker.upper()}})
        if filing_year:
            conditions.append({"filing_year": {"$eq": int(filing_year)}})
        if form_type:
            conditions.append({"form_type": {"$eq": form_type.upper()}})
        if not conditions:
            return None
        return conditions[0] if len(conditions) == 1 else {"$and": conditions}

    # ── Search methods ────────────────────────────────────────────────────────

    def bm25_search(
        self, query: str, top_k: int, mask: Optional[np.ndarray] = None
    ) -> List[RetrievedChunk]:
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        if mask is not None:
            scores = scores * mask
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [self._chunk_from_row(i, scores[i], "bm25") for i in top_idx if scores[i] > 0]

    def dense_search(
        self,
        query: str,
        top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        """ANN search via ChromaDB using a single query embedding (fast)."""
        q_emb = dense_model.encode([query], normalize_embeddings=True)[0].tolist()
        where = self._chroma_where(ticker, filing_year, form_type)
        kwargs: dict = {
            "query_embeddings": [q_emb],
            "n_results": top_k,
            "include": ["documents", "metadatas", "distances"],
        }
        if where:
            kwargs["where"] = where
        results = self.chroma_col.query(**kwargs)
        return [
            self._chunk_from_chroma(doc_id, text, meta, dist)
            for doc_id, text, meta, dist in zip(
                results["ids"][0],
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    def hybrid_search(
        self,
        query: str,
        bm25_top_k: int,
        dense_top_k: int,
        rerank_top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        mask         = self._bm25_mask(ticker, filing_year, form_type)
        bm25_results  = self.bm25_search(query, bm25_top_k, mask)
        dense_results = self.dense_search(query, dense_top_k, ticker, filing_year, form_type)

        # RRF merge — deduplicate on string chunk_id
        rrf: Dict[str, float] = {}
        pool: Dict[str, RetrievedChunk] = {}
        for rank, c in enumerate(bm25_results):
            rrf[c.chunk_id]  = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            pool[c.chunk_id] = c
        for rank, c in enumerate(dense_results):
            rrf[c.chunk_id] = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            if c.chunk_id not in pool:
                pool[c.chunk_id] = c

        candidates = [pool[k] for k in sorted(rrf, key=rrf.__getitem__, reverse=True)]
        if not candidates:
            return []

        # CrossEncoder rerank
        scores = reranker.predict(
            [(query, c.contextual_chunk) for c in candidates], show_progress_bar=False
        )
        for c, s in zip(candidates, scores):
            c.score  = float(s)
            c.source = "hybrid_reranked"

        return sorted(candidates, key=lambda x: x.score, reverse=True)[:rerank_top_k]


# ---------------------------------------------------------------------------
# Build global index — BM25 + ChromaDB load (~5s, no GPU/encoding needed)
# ---------------------------------------------------------------------------
print("Building global corpus index…")
chunk_dicts  = sec_df_to_chunk_dicts(sec_df)
global_index = CorpusIndex(chunk_dicts, chroma_path=CONFIG["chroma_db_path"])
print("Done.")

Building global corpus index…
  Building BM25 index…
  Loading ChromaDB collection…
  ChromaDB ready: 'sec_filings'  (12,606 docs)
  CorpusIndex ready: 12,606 BM25 chunks + ChromaDB dense index
Done.


In [8]:
def make_llm(model_name: str, temperature: float):
    if CONFIG["provider"] == "groq":
        return ChatGroq(model=model_name, temperature=temperature, groq_api_key=GROQ_API_KEY)
    elif CONFIG["provider"] == "gemini":
        return ChatGoogleGenerativeAI(model=model_name, temperature=temperature, google_api_key=GOOGLE_API_KEY)
    else:
        raise ValueError(f"Unknown provider: {CONFIG['provider']!r}")


def resolve_role_temperature(role: str, task_name: str = None) -> float:
    if task_name is not None:
        key = f"temperature_{task_name}"
        if key in CONFIG:
            return CONFIG[key]
    return CONFIG[f"temperature_{role}"] if f"temperature_{role}" in CONFIG else 0.0


def get_llm_candidates(role: str, task_name: str = None):
    temperature = resolve_role_temperature(role, task_name)
    return [
        (model_name, make_llm(model_name, temperature))
        for model_name in resolve_fallback_model_names(role)
    ]


_preferred_models_by_task: Dict[str, str] = {}


def order_model_candidates(role: str, task_name: str = None):
    candidates = get_llm_candidates(role, task_name)
    preference_key = task_name or role
    preferred = _preferred_models_by_task.get(preference_key)
    if not preferred:
        return candidates
    candidate_names = [name for name, _ in candidates]
    if preferred not in candidate_names:
        return candidates
    preferred_idx = candidate_names.index(preferred)
    return candidates[preferred_idx:]

In [9]:
# ---------------------------------------------------------------------------
# Pydantic output schemas
# ---------------------------------------------------------------------------

def _coerce_bool(v) -> bool:
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def _coerce_float(v) -> float:
    if isinstance(v, str):
        return float(v.strip())
    return float(v)


# ── Proposal 1: Query Decomposition ─────────────────────────────────────────

class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(
        default=None,
        description="Company ticker (e.g. AAPL, MSFT) if this sub-query is specific to one company. "
                    "None for cross-company sub-queries."
    )
    filing_year: Optional[int] = Field(
        default=None,
        description="Specific fiscal year (e.g. 2024) if this sub-query is year-specific. None otherwise."
    )
    form_type: Optional[str] = Field(
        default=None,
        description="10-K or 10-Q if the sub-query is specific to one form type. None otherwise."
    )

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts = []
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item) for item in fields))
            for key in ("query", "query_field", "table", "metric", "line_item", "fiscal_year"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True if the question compares multiple time periods, multiple companies, "
                    "or requires data from multiple distinct filings. "
                    "False for single-period, single-company questions."
    )
    sub_queries: List[SubQuery] = Field(
        description="If needs_decomposition=False: exactly one sub-query with the rewritten question. "
                    "If needs_decomposition=True: 2–3 focused sub-queries, one per time period or company."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    if isinstance(value, list):
                        normalized["sub_queries"] = value
                    else:
                        normalized["sub_queries"] = [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Standard schemas ─────────────────────────────────────────────────────────

class QueryRewriteOutput(BaseModel):
    rewritten_query: str = Field(description="Concise rewritten query optimized for retrieval")


class ContextEvalOutput(BaseModel):
    relevant: bool = Field(description="Whether the retrieved context is relevant and sufficient")
    reason: str = Field(description="Short reason")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "relevant" not in normalized and "is_relevant" in normalized:
            normalized["relevant"] = normalized["is_relevant"]
        answer_text = ""
        if "answer" in normalized:
            answer_text = str(normalized["answer"]).strip().lower()
        reason_text = str(normalized.get("reason", "")).strip().lower()
        combined_text = " ".join(part for part in (answer_text, reason_text) if part)
        if "relevant" not in normalized and "answer" in normalized:
            negative_markers = (
                "not available",
                "n/a",
                "not found",
                "insufficient data",
                "insufficient information",
                "not enough information",
                "cannot determine",
                "unable to determine",
                "cannot answer",
                "does not contain",
                "doesn't contain",
                "cannot infer",
                "unable to infer",
                "missing from the context",
                "not in the context",
            )
            if any(marker in combined_text for marker in negative_markers):
                normalized["relevant"] = False
        if "relevant" not in normalized and "answer" in normalized:
            if reason_text and any(marker in reason_text for marker in ("does not provide", "does not contain", "not provide", "not contain", "cannot determine", "not enough information", "insufficient information")):
                normalized["relevant"] = False
        if "relevant" not in normalized:
            signal_keys = {
                k for k in normalized.keys()
                if k not in {"reason", "is_sufficient", "answer", "is_relevant"}
            }
            if signal_keys:
                normalized["relevant"] = True
        if "reason" not in normalized:
            if "is_sufficient" in normalized:
                normalized["reason"] = f"Model returned is_sufficient={normalized['is_sufficient']}"
            elif "answer" in normalized:
                normalized["reason"] = f"Model returned answer={normalized['answer']}"
            else:
                normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("relevant", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Proposal 2: Refined Critic ───────────────────────────────────────────────

class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"] = Field(
        description=(
            "accept: the answer is grounded in the context (even if partial). "
            "repair: the answer has a fixable error — wrong period, wrong metric, wrong units, "
            "or directly contradicts the context. The data IS present but the answer is wrong. "
            "insufficient_data: the required financial data is completely absent from ALL retrieved "
            "chunks — not merely incomplete. Never use this just because the question is complex "
            "or the answer is approximate."
        )
    )
    reason: str = Field(description="Short critique")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("verdict", "label", "result", "status", "answer"):
            if "decision" not in normalized and alias in normalized:
                normalized["decision"] = normalized[alias]
                break
        if "decision" not in normalized:
            if _coerce_bool(normalized.get("repair")) is True:
                normalized["decision"] = "repair"
            elif _coerce_bool(normalized.get("accept")) is True:
                normalized["decision"] = "accept"
            elif _coerce_bool(normalized.get("insufficient_data")) is True:
                normalized["decision"] = "insufficient_data"
            elif _coerce_bool(normalized.get("refuse")) is True:
                normalized["decision"] = "insufficient_data"
        for alias in ("rationale", "explanation", "feedback", "critique", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if "reason" not in normalized:
            normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "approve": "accept",
                "approved": "accept",
                "pass": "accept",
                "grounded": "accept",
                "supported": "accept",
                "repair": "repair",
                "revise": "repair",
                "revised": "repair",
                "fix": "repair",
                "correct": "repair",
                "retry": "repair",
                "insufficient_data": "insufficient_data",
                "insufficient": "insufficient_data",
                "missing_data": "insufficient_data",
                "no_data": "insufficient_data",
                "not_enough_data": "insufficient_data",
                "cannot_answer": "insufficient_data",
                "unanswerable": "insufficient_data",
            }
            return mapping.get(raw, raw)
        return v


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"] = Field(
        description=(
            "Final decision after repair. Use 'accept' when you have a revised answer — "
            "even partial. Use 'warn' only for genuine ambiguity. "
            "Use 'refuse' ONLY if the question is entirely out of scope."
        )
    )
    revised_answer: str = Field(description="The final revised answer.")
    needs_new_retrieval: bool = Field(
        description="true or false — whether fresh retrieval would help. Must be a boolean, not a string."
    )
    reason: str = Field(description="Short rationale")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "data" in normalized and isinstance(normalized["data"], dict):
            for key, value in normalized["data"].items():
                normalized.setdefault(key, value)
        for alias in ("answer", "final_answer", "response"):
            if "revised_answer" not in normalized and alias in normalized:
                normalized["revised_answer"] = normalized[alias]
                break
        if "revised_answer" not in normalized:
            payload_keys = [
                k for k in normalized.keys()
                if k not in {"decision", "reason", "needs_new_retrieval", "data"}
            ]
            if payload_keys:
                normalized["revised_answer"] = "; ".join(
                    f"{k}={normalized[k]}" for k in payload_keys
                )
        for alias in ("rationale", "explanation", "feedback", "comment", "context"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if "needs_new_retrieval" not in normalized:
            normalized["needs_new_retrieval"] = False
        if "reason" not in normalized:
            if "revised_answer" in normalized:
                normalized["reason"] = "Model returned a revised answer"
            else:
                normalized["reason"] = "No reason provided"
        return normalized
    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "warn": "warn",
                "warning": "warn",
                "refuse": "refuse",
                "refusal": "refuse",
                "reject": "refuse",
            }
            return mapping.get(raw, raw)
        return v

    @field_validator("needs_new_retrieval", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


class JudgeOutput(BaseModel):
    score: float = Field(description="Overall correctness score — a number between 0.0 and 1.0, not a string")
    claims_covered: float = Field(
        description="Fraction of key facts covered — a number between 0.0 and 1.0, not a string"
    )
    reason: str = Field(description="Short explanation")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("overall_score", "correctness_score", "faithfulness_score", "grounding_score", "rating", "grade"):
            if "score" not in normalized and alias in normalized:
                normalized["score"] = normalized[alias]
                break
        for alias in ("coverage", "coverage_score", "supported_fraction", "support_rate", "claim_coverage"):
            if "claims_covered" not in normalized and alias in normalized:
                normalized["claims_covered"] = normalized[alias]
                break
        for alias in ("rationale", "explanation", "feedback", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if isinstance(normalized.get("reason"), dict):
            normalized["reason"] = "; ".join(f"{k}={v}" for k, v in normalized["reason"].items())
        if "claims_covered" not in normalized and "score" in normalized:
            normalized["claims_covered"] = normalized["score"]
        if "reason" not in normalized:
            normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("score", "claims_covered", mode="before")
    @classmethod
    def coerce_float(cls, v): return _coerce_float(v)

In [10]:
# ---------------------------------------------------------------------------
# Prompt templates
# ---------------------------------------------------------------------------

# ── Proposal 1: Planner ───────────────────────────────────────────────────────
planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial research planner working with SEC filings from these companies:\n"
        "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
        "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
        "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
        "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
        "Decompose (needs_decomposition=True) when the question:\n"
        "  • Explicitly compares two different fiscal years (e.g. 2023 vs 2024)\n"
        "  • Compares two different companies\n\n"
        "Do NOT decompose single-period, single-company questions.\n\n"
        "For each sub-query set ticker if company-specific, filing_year if year-specific.\n"
        "Every sub-query object must include a non-empty query field.\n"
        "filing_year = the calendar year the 10-K or 10-Q was filed "
        "(e.g. Apple FY2024 10-K was filed in November 2024, so filing_year=2024). "
        "Return a valid JSON object only that matches the requested schema."
    ),
    ("human", "Question: {question}"),
])

context_eval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
        "Mark as not relevant only if the context is clearly from the wrong company/period, or completely empty. "
        "Partial tables or incomplete passages still count as relevant if they contain the right metric. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

generator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial analyst answering questions using only the retrieved filing context. "
        "Be precise with numbers — include units (millions, billions, %), fiscal periods, and line item names. "
        "If the context does not contain the answer, say so explicitly rather than estimating.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

# ── Proposal 2: Updated critic prompt ────────────────────────────────────────
critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a critic for a financial RAG system. Evaluate the answer and choose one decision:\n\n"
        "  accept          — the answer is numerically grounded in the context (even if partial).\n"
        "  repair          — the answer has a fixable error: wrong period, wrong metric, wrong units, "
        "or contradicts the context. The data IS present but the answer used it incorrectly.\n"
        "  insufficient_data — the required financial data is completely absent from ALL retrieved "
        "chunks. Only use this when no amount of repair can help — the filing simply does not "
        "contain the information. Never use this just because the answer is approximate. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCurrent answer:\n{answer}"),
])

repair_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You repair a financial RAG answer after critique. Your default action is to ACCEPT with a revised answer. "
        "Pay close attention to: correct fiscal year/quarter, correct units (millions vs billions), "
        "correct sign, and the right line item name. "
        "Set decision='accept' whenever you can produce any useful revised answer — even partial. "
        "Set decision='warn' only if the context is genuinely ambiguous. "
        "Set decision='refuse' ONLY if the question is entirely off-topic for all retrieved filings. "
        "Set needs_new_retrieval=true only if critical data is completely absent from the context. "
        "Return a valid JSON object only that matches the requested schema."
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\n"
        "Original answer:\n{answer}\n\nCritic feedback:\n{critique}",
    ),
])

judge_correctness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
        "1 = correct value, correct units, correct period. "
        "0 = wrong number, wrong company, wrong period, or completely missing. "
        "Give partial credit for answers close but with unit errors. "
        "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    (
        "human",
        "Question:\n{question}\n\nReference answer:\n{reference_answer}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

judge_faithfulness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
        "1 = every number and claim is directly supported by the context. "
        "0 = answer contains numbers or claims not present in the context (hallucinated). "
        "Also set claims_covered: fraction of claims in the candidate supported by the context. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

In [11]:
# ---------------------------------------------------------------------------
# Rate limiter + safe LLM invocation helpers
# ---------------------------------------------------------------------------

class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window = 60.0
        self.timestamps: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self.timestamps and now - self.timestamps[0] >= self.window:
                self.timestamps.popleft()
            if len(self.timestamps) >= self.max_rpm:
                sleep_for = self.window - (now - self.timestamps[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] {self.max_rpm} RPM cap — waiting {sleep_for:.1f}s")
                    time.sleep(sleep_for)
            self.timestamps.append(time.time())


_rate_limiter = RateLimiter(max_rpm=CONFIG["max_rpm"])


def is_rate_limit_error(exc: Exception) -> bool:
    message = str(exc).lower()
    return "rate limit" in message or "rate_limit" in message or "429" in message


def safe_invoke_structured(
    role: str,
    schema_class,
    prompt,
    variables: Dict[str, Any],
    fallback: BaseModel,
    task_name: str = None,
) -> BaseModel:
    # Use json_mode to bypass Groq's strict tool-call schema validation.
    # The model outputs raw JSON which Pydantic then parses — field_validators
    # handle any string→bool/float coercion the model produces.
    model_candidates = order_model_candidates(role, task_name)
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, (model_name, llm) in enumerate(model_candidates):
        for attempt in range(max_attempts):
            try:
                _rate_limiter.wait()
                structured = llm.with_structured_output(schema_class, method="json_mode")
                result = (prompt | structured).invoke(variables)
                _preferred_models_by_task[preference_key] = model_name
                return result
            except Exception as e:
                print(
                    f"  [WARN] {schema_class.__name__} attempt {attempt+1}/{max_attempts} "
                    f"on {model_name} failed: {e}"
                )
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_model = model_candidates[model_idx + 1][0]
                    _preferred_models_by_task[preference_key] = next_model
                    print(f"  [WARN] Switching {role} model from {model_name} to {next_model} after rate limit.")
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        print(f"  [WARN] All retries exhausted for {schema_class.__name__}, using fallback.")
                        return fallback
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback


def safe_invoke_llm(
    role: str,
    prompt,
    variables: Dict[str, Any],
    fallback_text: str = "",
    task_name: str = None,
) -> str:
    model_candidates = order_model_candidates(role, task_name)
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, (model_name, llm) in enumerate(model_candidates):
        chain = prompt | llm
        for attempt in range(max_attempts):
            try:
                _rate_limiter.wait()
                result = chain.invoke(variables).content
                _preferred_models_by_task[preference_key] = model_name
                return result
            except Exception as e:
                print(f"  [WARN] LLM call attempt {attempt+1}/{max_attempts} on {model_name} failed: {e}")
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_model = model_candidates[model_idx + 1][0]
                    _preferred_models_by_task[preference_key] = next_model
                    print(f"  [WARN] Switching {role} model from {model_name} to {next_model} after rate limit.")
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        return fallback_text
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback_text


def format_retrieved_context(
    chunks: List[RetrievedChunk],
    max_chunks: int = None,
    max_chars: int = None,
) -> str:
    selected = list(chunks[: max_chunks or len(chunks)])
    parts = []
    total_chars = 0
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
        total_chars = sum(len(p) for p in parts) + max(0, 2 * (len(parts) - 1))
        if max_chars is not None and total_chars >= max_chars:
            joined = "\n\n".join(parts)
            return joined[:max_chars]
    return "\n\n".join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(query: str, ticker: str = None, filing_year: int = None, form_type: str = None) -> str:
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())
    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join(part for part in parts if part).strip()


def fails_retrieval_sanity_check(question: str, sub_queries: List[Dict[str, Any]], retrieved: List[RetrievedChunk]) -> bool:
    if not CONFIG.get("enable_retrieval_sanity_check", False):
        return False
    if not retrieved:
        return False
    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False
    present_tickers = {getattr(chunk, "ticker", None) for chunk in retrieved if getattr(chunk, "ticker", None)}
    if not present_tickers:
        return False
    return not expected_tickers.issubset(present_tickers)


def get_active_model_name(role: str, task_name: str = None) -> str:
    candidates = order_model_candidates(role, task_name)
    return candidates[0][0] if candidates else resolve_model_name(role)


def get_model_aware_context_limits(
    role: str,
    task_name: str,
    default_chunks: int,
    default_chars: int,
) -> Tuple[int, int]:
    model_name = get_active_model_name(role, task_name).lower()
    if "llama-3.1-8b-instant" in model_name:
        budgets = {
            "context_eval": (2, 2200),
            "critic": (2, 2500),
            "repair": (2, 2500),
            "judge": (2, 2200),
            "generator": (5, 7000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    if "qwen/qwen3-32b" in model_name:
        budgets = {
            "context_eval": (3, 3500),
            "critic": (3, 4000),
            "repair": (3, 4000),
            "judge": (3, 3200),
            "generator": (6, 9000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    return default_chunks, default_chars

In [12]:
# ---------------------------------------------------------------------------
# LangGraph state + nodes
# ---------------------------------------------------------------------------

class GraphState(TypedDict, total=False):
    question:              str
    index:                 CorpusIndex
    # Planner outputs
    rewritten_query:       str
    sub_queries:           List[Dict]   # list of SubQuery dicts
    needs_decomposition:   bool
    # Retrieval
    retrieved_chunks:      List[RetrievedChunk]
    retrieved_doc_names:   List[str]
    retrieval_sanity_failed: bool
    # Context eval
    context_retries:       int
    context_eval_relevant: bool
    # Generation
    generated_answer:      str
    # Critic
    critic_decision:       str
    critic_reason:         str
    # Repair
    repair_used:           bool
    repair_decision:       str
    repair_reason:         str
    needs_new_retrieval:   bool
    repair_retrieval_count: int
    is_repair_retrieval:   bool
    # Final
    final_answer:          str


# ── Proposal 1: Planner node ─────────────────────────────────────────────────

def node_query_planner(state: GraphState) -> GraphState:
    result = safe_invoke_structured(
        "agent",
        PlannerOutput,
        planner_prompt,
        {"question": state["question"]},
        PlannerOutput(
            needs_decomposition=False,
            sub_queries=[SubQuery(query=state["question"])],
        ),
        task_name="planner",
    )
    sub_queries = []
    for sq in result.sub_queries:
        sq_dict = sq.model_dump()
        query = (sq_dict.get("query") or "").strip()
        if not query:
            parts = [state["question"]]
            if sq_dict.get("ticker"):
                parts.append(f"ticker {sq_dict['ticker']}")
            if sq_dict.get("filing_year"):
                parts.append(f"filing year {sq_dict['filing_year']}")
            if sq_dict.get("form_type"):
                parts.append(f"form {sq_dict['form_type']}")
            query = " | ".join(parts)
        sq_dict["query"] = cleanup_planner_query(
            query,
            ticker=sq_dict.get("ticker"),
            filing_year=sq_dict.get("filing_year"),
            form_type=sq_dict.get("form_type"),
        )
        sub_queries.append(sq_dict)
    rewritten_query = sub_queries[0]["query"] if sub_queries else state["question"]
    n = len(sub_queries)
    label = "decomposed" if result.needs_decomposition else "single"
    print(f"  [Planner] {label} | {n} sub-quer{'ies' if n > 1 else 'y'}")
    if result.needs_decomposition:
        for sq in sub_queries:
            print(f"    → {sq['query']}  (ticker={sq.get('ticker')}, year={sq.get('filing_year')})")
    return {
        "rewritten_query":     rewritten_query,
        "sub_queries":         sub_queries,
        "needs_decomposition": result.needs_decomposition,
    }


# ── Updated hybrid retriever (multi-query aware) ─────────────────────────────

def node_hybrid_retriever(state: GraphState) -> GraphState:
    sub_queries = state.get("sub_queries", [])

    if len(sub_queries) <= 1:
        # Single-query path
        query = state.get("rewritten_query", state["question"])
        sq = sub_queries[0] if sub_queries else {}
        retrieved = state["index"].hybrid_search(
            query,
            bm25_top_k=CONFIG["bm25_top_k"],
            dense_top_k=CONFIG["dense_top_k"],
            rerank_top_k=CONFIG["rerank_top_k"],
            ticker=sq.get("ticker"),
            filing_year=sq.get("filing_year"),
            form_type=sq.get("form_type"),
        )
    else:
        # Multi-query decomposition: retrieve per sub-query then merge by best score
        per_k = CONFIG["decomposition_top_k_per_subquery"]
        merged: Dict[int, RetrievedChunk] = {}
        for sq in sub_queries:
            chunks = state["index"].hybrid_search(
                sq["query"],
                bm25_top_k=CONFIG["bm25_top_k"],
                dense_top_k=CONFIG["dense_top_k"],
                rerank_top_k=per_k,
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
            for chunk in chunks:
                key = chunk.chunk_id
                if key not in merged or chunk.score > merged[key].score:
                    merged[key] = chunk
        retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[:CONFIG["rerank_top_k"]]

    sanity_failed = fails_retrieval_sanity_check(state["question"], sub_queries, retrieved)
    return {
        "retrieved_chunks":         retrieved,
        "retrieved_doc_names":      extract_retrieved_doc_names(retrieved),
        "retrieval_sanity_failed":  sanity_failed,
    }


def node_context_evaluator(state: GraphState) -> GraphState:
    if state.get("retrieval_sanity_failed", False):
        return {"context_eval_relevant": False, "context_retries": state.get("context_retries", 0)}
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "context_eval", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        ContextEvalOutput,
        context_eval_prompt,
        {"question": state["question"], "context": context},
        ContextEvalOutput(relevant=True, reason="Fallback accept"),
        task_name="context_eval",
    )
    return {"context_eval_relevant": result.relevant, "context_retries": state.get("context_retries", 0)}


def route_context(state: GraphState) -> str:
    if state.get("context_eval_relevant", True):
        return "generator"
    if state.get("context_retries", 0) < CONFIG["max_context_retries"]:
        return "retry_retrieval"
    return "generator"


def node_increment_context_retry(state: GraphState) -> GraphState:
    return {"context_retries": state.get("context_retries", 0) + 1}


def node_generator(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    raw = safe_invoke_llm(
        "generator",
        generator_prompt,
        {"question": state["question"], "context": context},
        task_name="generator",
    )
    answer = normalize_answer_text(raw)
    return {"generated_answer": answer, "final_answer": answer}


# ── Proposal 2: Updated critic node + routing ─────────────────────────────────

def node_critic(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "critic", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        CriticOutput,
        critic_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
        },
        CriticOutput(decision="accept", reason="Fallback accept"),
        task_name="critic",
    )
    updates: GraphState = {"critic_decision": result.decision, "critic_reason": result.reason}
    # Proposal 2: if data is absent, set the final answer immediately so we exit cleanly
    if result.decision == "insufficient_data":
        updates["final_answer"] = (
            f"Insufficient data: The required information could not be found in the "
            f"retrieved SEC filings. ({result.reason})"
        )
    return updates


def route_critic(state: GraphState) -> str:
    if state.get("is_repair_retrieval", False):
        return "end"
    decision = state.get("critic_decision", "accept")
    if decision == "repair":
        return "repair"
    # Both 'accept' and 'insufficient_data' exit — final_answer already set
    return "end"


def node_repair(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "repair", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        RepairOutput,
        repair_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
            "critique": state.get("critic_reason", "Needs repair"),
        },
        RepairOutput(
            decision="accept",
            revised_answer=state.get("generated_answer", ""),
            needs_new_retrieval=False,
            reason="Fallback repair",
        ),
        task_name="repair",
    )
    count = state.get("repair_retrieval_count", 0)
    if result.needs_new_retrieval:
        count += 1
    return {
        "repair_used":           True,
        "repair_decision":       result.decision,
        "repair_reason":         result.reason,
        "needs_new_retrieval":   result.needs_new_retrieval,
        "repair_retrieval_count": count,
        "final_answer":          normalize_answer_text(result.revised_answer),
    }


def route_repair(state: GraphState) -> str:
    if state.get("needs_new_retrieval", False) and state.get("repair_retrieval_count", 0) <= CONFIG["max_repair_retrievals"]:
        return "re_retrieve"
    return "end"


def node_mark_repair_retrieval(state: GraphState) -> GraphState:
    return {"is_repair_retrieval": True}

In [13]:
# ---------------------------------------------------------------------------
# Graph construction
# Entry point changed: query_planner replaces query_rewriter
# ---------------------------------------------------------------------------

def build_agentic_graph():
    g = StateGraph(GraphState)

    g.add_node("query_planner",          node_query_planner)
    g.add_node("hybrid_retriever",        node_hybrid_retriever)
    g.add_node("context_evaluator",       node_context_evaluator)
    g.add_node("increment_context_retry", node_increment_context_retry)
    g.add_node("generator",               node_generator)
    g.add_node("critic",                  node_critic)
    g.add_node("repair",                  node_repair)
    g.add_node("mark_repair_retrieval",   node_mark_repair_retrieval)

    g.set_entry_point("query_planner")
    g.add_edge("query_planner",          "hybrid_retriever")
    g.add_edge("hybrid_retriever",        "context_evaluator")
    g.add_conditional_edges(
        "context_evaluator",
        route_context,
        {"generator": "generator", "retry_retrieval": "increment_context_retry"},
    )
    g.add_edge("increment_context_retry", "hybrid_retriever")
    g.add_edge("generator",               "critic")
    g.add_conditional_edges(
        "critic",
        route_critic,
        {"repair": "repair", "end": END},
    )
    g.add_conditional_edges(
        "repair",
        route_repair,
        {"re_retrieve": "mark_repair_retrieval", "end": END},
    )
    g.add_edge("mark_repair_retrieval",   "hybrid_retriever")

    return g.compile()


agentic_graph = build_agentic_graph()
print("Agentic graph compiled.")

Agentic graph compiled.


In [14]:
# ---------------------------------------------------------------------------
# Metrics helpers
# ---------------------------------------------------------------------------

def normalize_answer_text(answer: Any) -> str:
    if answer is None:
        return ""
    if isinstance(answer, str):
        return answer.strip()
    if isinstance(answer, list):
        return " ".join(str(x) for x in answer if x is not None).strip()
    if isinstance(answer, dict):
        for key in ["final_answer", "answer", "text", "content", "generated_answer"]:
            if key in answer:
                return normalize_answer_text(answer[key])
        return str(answer).strip()
    return str(answer).strip()


def normalize_for_eval(text: Any) -> str:
    text = normalize_answer_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_f1_score(prediction: Any, reference: Any) -> float:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    p = n_same / len(pred_tokens)
    r = n_same / len(ref_tokens)
    return 2 * p * r / (p + r)


def _extract_numbers(text: str) -> List[float]:
    text = text.lower()
    for word, mult in [("trillion", 1e12), ("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]:
        text = re.sub(
            rf"([\d,.]+)\s*{word}",
            lambda m, mult=mult: str(float(m.group(1).replace(",", "")) * mult),
            text,
        )
    numbers = []
    for tok in re.findall(r"\(?\$?[\d,]+\.?\d*\)?", text):
        negative = tok.startswith("(") and tok.endswith(")")
        try:
            val = float(re.sub(r"[^\d.]", "", tok))
            numbers.append(-val if negative else val)
        except ValueError:
            pass
    return numbers


def numerical_accuracy_score(prediction: Any, reference: Any, tolerance: float = 0.01) -> float:
    pred_nums = _extract_numbers(normalize_answer_text(prediction))
    ref_nums  = _extract_numbers(normalize_answer_text(reference))
    if not ref_nums:
        return 1.0
    if not pred_nums:
        return 0.0
    for ref_val in ref_nums:
        for pred_val in pred_nums:
            if ref_val == 0 and pred_val == 0:
                return 1.0
            if ref_val != 0 and abs(pred_val - ref_val) / abs(ref_val) <= tolerance:
                return 1.0
    return 0.0


def compute_decision_from_text(answer: Any) -> str:
    lower = normalize_answer_text(answer).lower()
    if any(x in lower for x in ["insufficient data", "cannot answer", "not available", "not found", "do not know"]):
        return "warn"
    if any(x in lower for x in ["refuse", "cannot comply"]):
        return "refuse"
    return "answer"

In [15]:
# ---------------------------------------------------------------------------
# Pipeline runners — all now accept the global index, no per-question corpus
# ---------------------------------------------------------------------------

def run_simple_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.dense_search(question, top_k=CONFIG["rerank_top_k"])
    gen_chunks, gen_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context   = format_retrieved_context(
        retrieved,
        max_chunks=gen_chunks,
        max_chars=gen_chars,
    )
    raw       = safe_invoke_llm(
        "generator", generator_prompt, {"question": question, "context": context}, task_name="generator"
    )
    answer    = normalize_answer_text(raw)
    return {
        "pipeline": "simple_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0, "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None, "repair_used": False, "repair_decision": None,
        "final_answer": answer, "latency_sec": time.time() - start,
        "needs_decomposition": False, "repair_retrieval_count": 0,
    }


def run_advanced_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.hybrid_search(
        question,
        bm25_top_k=CONFIG["bm25_top_k"],
        dense_top_k=CONFIG["dense_top_k"],
        rerank_top_k=CONFIG["rerank_top_k"],
    )
    gen_chunks, gen_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context = format_retrieved_context(
        retrieved,
        max_chunks=gen_chunks,
        max_chars=gen_chars,
    )
    raw     = safe_invoke_llm(
        "generator", generator_prompt, {"question": question, "context": context}, task_name="generator"
    )
    answer  = normalize_answer_text(raw)
    return {
        "pipeline": "advanced_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0, "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None, "repair_used": False, "repair_decision": None,
        "final_answer": answer, "latency_sec": time.time() - start,
        "needs_decomposition": False, "repair_retrieval_count": 0,
    }


def run_agentic_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    state: GraphState = {
        "question":              question,
        "index":                 index,
        "context_retries":       0,
        "repair_used":           False,
        "repair_retrieval_count": 0,
        "is_repair_retrieval":   False,
        "retrieval_sanity_failed": False,
        "sub_queries":           [],
        "needs_decomposition":   False,
    }
    out     = agentic_graph.invoke(state)
    latency = time.time() - start

    generated = normalize_answer_text(out.get("generated_answer", ""))
    final     = normalize_answer_text(out.get("final_answer", generated))
    return {
        "pipeline":              "agentic_rag",
        "rewritten_query":       normalize_answer_text(out.get("rewritten_query", question)),
        "retrieved_chunks":      out.get("retrieved_chunks", []),
        "retrieved_doc_names":   out.get("retrieved_doc_names", []),
        "context_retries":       out.get("context_retries", 0),
        "context_eval_relevant": out.get("context_eval_relevant"),
        "generated_answer":      generated,
        "critic_decision":       out.get("critic_decision"),
        "repair_used":           out.get("repair_used", False),
        "repair_decision":       out.get("repair_decision"),
        "repair_retrieval_count": out.get("repair_retrieval_count", 0),
        "needs_decomposition":   out.get("needs_decomposition", False),
        "final_answer":          final,
        "latency_sec":           latency,
    }

In [16]:
# ---------------------------------------------------------------------------
# Evaluation loop
# ---------------------------------------------------------------------------

def run_all_pipelines(eval_df: pd.DataFrame, index: CorpusIndex) -> pd.DataFrame:
    rows = []
    sleep_sec = CONFIG["inter_question_sleep_sec"]

    for q_idx, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df), desc="Questions")):
        qid              = row["id"]
        question         = row["question"]
        reference_answer = row.get("reference_answer", "")
        company          = row.get("company", "")
        question_type    = row.get("question_type", "unknown")
        expected_decision = row.get("expected_decision", "answer")

        pipeline_outputs = [
            run_simple_rag(question, index),
            run_advanced_rag(question, index),
            run_agentic_rag(question, index),
        ]

        for out in pipeline_outputs:
            final_answer     = normalize_answer_text(out.get("final_answer", ""))
            generated_answer = normalize_answer_text(out.get("generated_answer", ""))

            predicted_decision = compute_decision_from_text(final_answer)
            decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

            # Token F1 and numerical accuracy — meaningful only when reference_answer is non-empty
            t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
            num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None

            rows.append({
                "id":                     qid,
                "question":               question,
                "company":                company,
                "question_type":          question_type,
                "reference_answer":       reference_answer,
                "expected_decision":      expected_decision,
                "pipeline":               out["pipeline"],
                "rewritten_query":        out.get("rewritten_query", question),
                "retrieved_doc_names":    out.get("retrieved_doc_names", []),
                "context_retries":        out.get("context_retries", 0),
                "context_eval_relevant":  out.get("context_eval_relevant"),
                "generated_answer":       generated_answer,
                "critic_decision":        out.get("critic_decision"),
                "repair_used":            out.get("repair_used", False),
                "repair_decision":        out.get("repair_decision"),
                "repair_retrieval_count": out.get("repair_retrieval_count", 0),
                "needs_decomposition":    out.get("needs_decomposition", False),
                "final_answer":           final_answer,
                "latency_sec":            out.get("latency_sec"),
                "token_f1":               t_f1,
                "numerical_accuracy":     num_acc,
                "predicted_decision":     predicted_decision,
                "decision_accuracy":      decision_accuracy,
                # Filled in by LLM judge later
                "llm_correctness_score":  None,
                "llm_correctness_reason": None,
                "llm_claims_covered":     None,
                "llm_faithfulness_score": None,
                "llm_faithfulness_reason":None,
                "retrieved_chunks":       out.get("retrieved_chunks", []),
            })

        if q_idx < len(eval_df) - 1:
            time.sleep(sleep_sec)

    return pd.DataFrame(rows)


results_df = run_all_pipelines(eval_df, global_index)

Questions:   0%|          | 0/15 [00:00<?, ?it/s]

  [WARN] LLM call attempt 1/3 on meta-llama/llama-4-scout-17b-16e-instruct failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 498947, Requested 2601. Please try again in 4m27.494399999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] Switching generator model from meta-llama/llama-4-scout-17b-16e-instruct to llama-3.1-8b-instant after rate limit.
  [Planner] single | 1 sub-query
  [Planner] single | 1 sub-query
  [Planner] single | 1 sub-query
  [WARN] LLM call attempt 1/3 on llama-3.1-8b-instant failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 

In [17]:
# ---------------------------------------------------------------------------
# LLM judging (faithfulness + correctness on a sample per pipeline)
# ---------------------------------------------------------------------------

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_correctness_prompt,
        {"question": question, "reference_answer": reference_answer, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason


def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_faithfulness_prompt,
        {"question": question, "context": context, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason


def run_llm_judging(
    results_df: pd.DataFrame,
    sample_n: int = CONFIG["judge_sample_n"],
    random_state: int = CONFIG["random_seed"],
) -> pd.DataFrame:
    if sample_n == 0:
        return results_df.copy()

    df = results_df.copy()
    rng = random.Random(random_state)

    for pipeline in df["pipeline"].unique():
        pipe_idx = df[df["pipeline"] == pipeline].index.tolist()
        sampled  = rng.sample(pipe_idx, min(sample_n, len(pipe_idx)))
        print(f"Judging {len(sampled)} questions for [{pipeline}]…")

        for idx in tqdm(sampled, desc=pipeline, leave=False):
            row = df.loc[idx]
            reference = row["reference_answer"]

            # Correctness (only meaningful when we have a reference answer)
            if reference:
                c_score, c_claims, c_reason = llm_judge_correctness(
                    row["question"], reference, row["final_answer"]
                )
                df.at[idx, "llm_correctness_score"]  = c_score
                df.at[idx, "llm_correctness_reason"] = c_reason
                df.at[idx, "llm_claims_covered"]     = c_claims

            # Faithfulness (always run — uses retrieved context)
            chunks = row.get("retrieved_chunks") if "retrieved_chunks" in df.columns else []
            judge_chunks, judge_chars = get_model_aware_context_limits(
                "judge", "judge", CONFIG["judge_max_context_chunks"], CONFIG["judge_max_context_chars"]
            )
            context_str = format_retrieved_context(
                chunks,
                max_chunks=judge_chunks,
                max_chars=judge_chars,
            ) if chunks and isinstance(chunks, list) and len(chunks) > 0 else row["final_answer"]
            f_score, _, f_reason = llm_judge_faithfulness(
                row["question"], context_str, row["final_answer"]
            )
            df.at[idx, "llm_faithfulness_score"]  = f_score
            df.at[idx, "llm_faithfulness_reason"] = f_reason

            time.sleep(CONFIG["inter_question_sleep_sec"])

    return df


results_df = run_llm_judging(results_df)

Judging 2 questions for [simple_rag]…


simple_rag:   0%|          | 0/2 [00:00<?, ?it/s]

Judging 2 questions for [advanced_rag]…


advanced_rag:   0%|          | 0/2 [00:00<?, ?it/s]

  [WARN] JudgeOutput attempt 1/3 on llama-3.1-8b-instant failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499809, Requested 1300. Please try again in 3m11.6352s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] Switching judge model from llama-3.1-8b-instant to qwen/qwen3-32b after rate limit.
Judging 2 questions for [agentic_rag]…


agentic_rag:   0%|          | 0/2 [00:00<?, ?it/s]

In [18]:
# ---------------------------------------------------------------------------
# Metrics aggregation + display
# ---------------------------------------------------------------------------

def nanmean(series) -> float:
    vals = [v for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def rate_of(series, match_val) -> float:
    vals = [v == match_val for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pipeline, grp in df.groupby("pipeline"):
        agentic = grp if pipeline == "agentic_rag" else None
        rows.append({
            "pipeline":                  pipeline,
            "n_questions":               len(grp),
            "token_f1":                  nanmean(grp["token_f1"]),
            "numerical_accuracy":        nanmean(grp["numerical_accuracy"]),
            "decision_accuracy":         nanmean(grp["decision_accuracy"]),
            "avg_latency_sec":           nanmean(grp["latency_sec"]),
            "llm_faithfulness_score":    nanmean(grp["llm_faithfulness_score"]),
            "llm_correctness_score":     nanmean(grp["llm_correctness_score"]),
            "llm_claims_covered":        nanmean(grp["llm_claims_covered"]),
            # Agentic-only metrics
            "decomposition_rate":        nanmean(grp["needs_decomposition"]) if "needs_decomposition" in grp else float("nan"),
            "critic_repair_rate":        rate_of(grp["critic_decision"], "repair"),
            "critic_insufficient_rate":  rate_of(grp["critic_decision"], "insufficient_data"),
            "repair_invocation_rate":    nanmean(grp["repair_used"]),
            "avg_context_retries":       nanmean(grp["context_retries"]),
        })
    return pd.DataFrame(rows).set_index("pipeline")


summary_df = aggregate_metrics(results_df)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
print("\n=== Summary metrics by pipeline ===")
display(summary_df.T)

# Show agentic-only per-question detail
agentic_rows = results_df[results_df["pipeline"] == "agentic_rag"].copy()
display_cols = [
    "id", "question_type", "needs_decomposition", "critic_decision",
    "repair_used", "llm_faithfulness_score", "llm_correctness_score",
    "final_answer",
]
print("\n=== Agentic RAG — per-question detail ===")
display(agentic_rows[display_cols].reset_index(drop=True))


=== Summary metrics by pipeline ===


pipeline,advanced_rag,agentic_rag,simple_rag
n_questions,15.000,15.000,15.000
token_f1,0.015,0.073,0.023
numerical_accuracy,0.200,0.400,0.200
decision_accuracy,0.933,0.667,0.933
avg_latency_sec,24.005,70.497,18.430
llm_faithfulness_score,0.500,1.000,0.500
llm_correctness_score,0.000,NaN,0.000
llm_claims_covered,0.000,NaN,0.000
decomposition_rate,0.000,0.600,0.000
critic_repair_rate,NaN,0.333,NaN



=== Agentic RAG — per-question detail ===


,id,question_type,needs_decomposition,critic_decision,repair_used,llm_faithfulness_score,llm_correctness_score,final_answer
0,SEC_001,single-period,False,repair,True,None,None,"Based on the provided context, we can see that..."
1,SEC_002,single-period,False,repair,True,None,None,"$245,122 million"
2,SEC_003,single-period,False,repair,True,None,None,The total revenue for fiscal year 2025 is $137...
3,SEC_004,single-period,False,insufficient_data,False,1.000,None,Insufficient data: The required information co...
4,SEC_005,single-period,False,insufficient_data,False,1.000,None,Insufficient data: The required information co...
5,SEC_006,single-period,False,accept,False,None,None,"<think>\nOkay, let's tackle this question: Wha..."
6,SEC_007,multi-period,True,accept,False,None,None,"<think>\nOkay, let's tackle this question abou..."
7,SEC_008,multi-period,True,repair,True,None,None,NVIDIA's Data Center revenue for fiscal year 2...
8,SEC_009,multi-period,True,accept,False,None,None,"<think>\nOkay, let's tackle this question abou..."
9,SEC_010,multi-period,True,insufficient_data,False,None,None,Insufficient data: The required information co...
